# AR Rule Consistency Batch Experiment

Run the dataset in metadata order. Set `x` to the number of existing completed records and `n_samples` to the maximum number of new images for this run. `x=0` starts from the first sample. If either result file contains more than `x` records, only the first `x` records are kept; if it contains fewer than `x`, the notebook raises an error. When the count equals `x`, processing continues from the next sample.

After every successful image, the notebook prints its binary match, descriptive match, tool-call counts, and end-to-end latency in seconds. It then incrementally saves two JSON files:
- `experiment_core_results.json`: file name, binary/descriptive matches, tool counts, and latency.
- `experiment_raw_results.json`: file name, full Agent output, and tool trace.

No full images or SAM masks are saved by this batch workflow.

In [ ]:
import json
import sys
import time
from collections import Counter
from pathlib import Path

from dotenv import load_dotenv

workspace_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "rule_agent_github").is_dir()
)
experiment_dir = workspace_root / "rule_agent_github"
load_dotenv(experiment_dir / ".env")
for search_path in ("", str(experiment_dir)):
    while search_path in sys.path:
        sys.path.remove(search_path)
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

for module_name in list(sys.modules):
    if module_name == "agents" or module_name.startswith("agents."):
        del sys.modules[module_name]
    if module_name == "rule_agent_github" or module_name.startswith("rule_agent_github."):
        del sys.modules[module_name]

from rule_agent_github.agents import create_agent
from rule_agent_github.configs import ExperimentConfig
from rule_agent_github.data import load_metadata
from rule_agent_github.pipeline import run_experiment
from rule_agent_github.tools import clear_debug_crops

# x is the checkpoint count; n_samples is the limit for this run.
x = 0
n_samples = 108
if not 0 <= x <= 108:
    raise ValueError("x must be between 0 and 108")
if not 0 <= n_samples <= 108 - x:
    raise ValueError("n_samples must be between 0 and 108 - x")

MODEL = "gpt-5.6-sol"
THINKING = "medium"
ANALYSIS_MODEL = "gpt-5.6-luna"
MAX_TURNS = 30
CORE_RESULTS_PATH = experiment_dir / "sol_experiment_core_results.json"
RAW_RESULTS_PATH = experiment_dir / "sol_experiment_raw_results.json"

agent = create_agent(MODEL, THINKING)
print("Agent:", agent.name)
print("Model:", agent.model)
print("Max turns:", MAX_TURNS)
print("Checkpoint x:", x)
print("Samples this run:", n_samples)
print("Tools:", [tool.name for tool in agent.tools])

In [ ]:
from openai import OpenAI


def load_results(path):
    if not path.exists():
        return []
    with path.open(encoding="utf-8") as result_file:
        value = json.load(result_file)
    if not isinstance(value, list):
        raise ValueError(f"Result file must contain a JSON list: {path}")
    return value


def save_results(path, records):
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    temporary_path.write_text(
        json.dumps(records, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    temporary_path.replace(path)


def analyze_agent_output(record, agent_output, tool_counts):
    tool_counts_text = "\n".join(
        f"{name}: {count}" for name, count in sorted(tool_counts.items())
    ) or "No tools called: 0"
    prompt = f"""
Compare the Agent output with the reference annotation. Return one valid JSON
object only with exactly these keys: prediction, inconsistency_tp,
inconsistency_fp, inconsistency_fn, and report. Prediction must be exactly
CONSISTENT or INCONSISTENT. The three inconsistency values must be non-negative
integers.

The TP/FP/FN are case-level counts for inconsistency prediction inside this
image, not image-level counts and not generic description-quality scores. A
case means one distinct rule-violation event reported by the Agent.

First extract the distinct violation cases from the Agent output and the
reference descriptive annotation. For a Consistent reference image, there are
zero reference violation cases: every violation case reported by the Agent is
FP, with TP=0 and FN=0. For Omission, Commission, or Confusion images, match
Agent-reported violation cases to reference violation cases by semantic
compatibility. A broader description or a different name is acceptable when it
clearly refers to the same violation case. Matched cases are TP, Agent-only
cases are FP, and reference-only cases are FN. If a case is not reported, it is
not a TP. Do not collapse multiple distinct violation cases into one just
because they share the same inconsistency type.

Determine the prediction field from the Agent's explicit overall conclusion,
especially its final conclusion or overall conclusion section. Do not infer a
CONSISTENT prediction merely because some entities are consistent, and do not
ignore an explicit INCONSISTENT conclusion. The prediction field is the
Agent's overall binary conclusion, while the three counts measure the
individual inconsistency cases behind that conclusion. The prediction and the
report must agree: if the Agent's overall conclusion says INCONSISTENT, return
prediction=INCONSISTENT; if it says CONSISTENT, return prediction=CONSISTENT.
Do not silently rewrite the Agent's conclusion based on your own judgment.

Reference inconsistency type: {record["Inconsistency Type"]}
Reference binary annotation: {record["Binary Annotation"]}
Reference descriptive annotation: {record["Descriptive Annotation"]}
Agent output:
{agent_output}

The report must briefly state the overall binary comparison and how individual
violation cases were matched and counted. The exact tool counts were:
{tool_counts_text}
"""
    response = OpenAI().responses.create(model=ANALYSIS_MODEL, input=prompt)
    text = response.output_text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    analysis = json.loads(text)
    prediction = analysis.get("prediction")
    if prediction not in {"CONSISTENT", "INCONSISTENT"}:
        raise ValueError(f"Invalid binary prediction: {prediction!r}")
    count_keys = ("inconsistency_tp", "inconsistency_fp", "inconsistency_fn")
    counts = [analysis.get(key) for key in count_keys]
    if not all(isinstance(value, int) and not isinstance(value, bool) and value >= 0 for value in counts):
        raise TypeError("GPT analysis must return non-negative integer case counts")
    return prediction, {
        "TP": counts[0],
        "FP": counts[1],
        "FN": counts[2],
    }


In [ ]:
metadata_records = load_metadata()
metadata_file_names = [item["Image Name"] for item in metadata_records]
core_results = load_results(CORE_RESULTS_PATH)
raw_results = load_results(RAW_RESULTS_PATH)
core_by_file = {item["file_name"]: item for item in core_results}
raw_by_file = {item["file_name"]: item for item in raw_results}

checkpoint_names = metadata_file_names[:x]
if len(core_by_file) < x or len(raw_by_file) < x:
    raise RuntimeError(
        f"Checkpoint x={x} requires at least x records in both result files; "
        f"found core={len(core_by_file)}, raw={len(raw_by_file)}."
    )

core_by_file = {
    name: core_by_file[name]
    for name in checkpoint_names
    if name in core_by_file
}
raw_by_file = {
    name: raw_by_file[name]
    for name in checkpoint_names
    if name in raw_by_file
}
if set(core_by_file) != set(checkpoint_names) or set(raw_by_file) != set(checkpoint_names):
    raise RuntimeError(
        "The first x metadata records are not present in both result files. "
        "Set x to a valid completed-record checkpoint."
    )
for name, item in core_by_file.items():
    counts = item.get("inconsistency_cases")
    if not isinstance(counts, dict) or set(counts) != {"TP", "FP", "FN"}:
        raise RuntimeError(
            f"Existing Agent result for {name} uses the old schema. "
            "Set x=0 to regenerate case-level TP/FP/FN results."
        )

save_results(CORE_RESULTS_PATH, list(core_by_file.values()))
save_results(RAW_RESULTS_PATH, list(raw_by_file.values()))
existing_files = set(checkpoint_names)

print(f"Checkpoint records kept: {x}")
print(f"Next sample: {metadata_file_names[x] if x < len(metadata_file_names) else 'none; all samples complete'}")

In [ ]:
failed_files = []
processed_this_run = 0

# Clear previous batches once; keep crops generated by this batch for debugging.
clear_debug_crops()

for metadata_index, record in enumerate(
    metadata_records[x:x + n_samples],
    start=x + 1,
):
    file_name = record["Image Name"]
    if file_name in existing_files:
        continue

    print(f"[{metadata_index}/108] Running {file_name}")
    start_time = time.perf_counter()
    try:
        config = ExperimentConfig(
            scene=int(record["SceneID"]),
            rule=int(record["Rule ID"]),
            inconsistency_type=record["Inconsistency Type"],
            model=MODEL,
            thinking=THINKING,
        )
        experiment_result = await run_experiment(
            config,
            agent=agent,
            max_turns=MAX_TURNS,
        )
        latency_seconds = round(time.perf_counter() - start_time, 3)
        tool_counts = dict(
            Counter(trace["tool_name"] for trace in experiment_result["tool_trace"])
        )
        prediction, inconsistency_cases = analyze_agent_output(
            record,
            experiment_result["agent_output"],
            tool_counts,
        )
        binary_annotation = int(record["Binary Annotation"]) == 1
        binary_match = (prediction == "INCONSISTENT") == binary_annotation

        core_result = {
            "file_name": file_name,
            "binary_match": binary_match,
            "inconsistency_cases": inconsistency_cases,
            "tool_counts": tool_counts,
            "latency_seconds": latency_seconds,
        }
        raw_result = {
            "file_name": file_name,
            "agent_results": experiment_result["agent_output"],
            "tool_trace": experiment_result["tool_trace"],
        }
        core_by_file[file_name] = core_result
        raw_by_file[file_name] = raw_result
        save_results(CORE_RESULTS_PATH, list(core_by_file.values()))
        save_results(RAW_RESULTS_PATH, list(raw_by_file.values()))
        existing_files.add(file_name)
        processed_this_run += 1

        print(
            f"Finished {file_name}: binary={binary_match}, "
            f"inconsistency_cases={inconsistency_cases}, "
            f"tool_calls={tool_counts}, latency={latency_seconds:.3f}s"
        )
    except Exception as error:
        failed_files.append({"file_name": file_name, "error": repr(error)})
        print(f"FAILED {file_name}: {error}")

print(f"Completed this run: {processed_this_run}")
print(f"Failed files in this run: {len(failed_files)}")

In [ ]:
final_core_results = load_results(CORE_RESULTS_PATH)
final_raw_results = load_results(RAW_RESULTS_PATH)
print("Core records:", len(final_core_results))
print("Raw records:", len(final_raw_results))
print("Binary matches:", sum(item["binary_match"] for item in final_core_results))
print("Case TP:", sum(item["inconsistency_cases"]["TP"] for item in final_core_results))
print("Case FP:", sum(item["inconsistency_cases"]["FP"] for item in final_core_results))
print("Case FN:", sum(item["inconsistency_cases"]["FN"] for item in final_core_results))
print("Failed files in this run:", failed_files)